In [ ]:
# 1. Instalar dependencias (dos pasos para evitar conflictos con requests)
!pip -q install -U fastapi uvicorn pyngrok langchain langchain-openrouter langchain-text-splitters

In [ ]:
!pip -q install requests==2.32.4 --upgrade --force-reinstall --no-deps

**Nota:** Si al instalar requests te da un error, vuelve a ejecutar la celda.

Imports y configuración de secretos:

In [ ]:
# 2. Importar librerías
import os

from fastapi import FastAPI, HTTPException, Body, Query
from pydantic import BaseModel, Field
from typing import List
from google.colab import userdata
from langchain_openrouter import ChatOpenRouter
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Configurar OpenRouter
openrouter_api_key = userdata.get("OPENROUTER_API_KEY")
if not openrouter_api_key:
    raise ValueError("Agrega OPENROUTER_API_KEY en los Secrets de Colab.")
os.environ["OPENROUTER_API_KEY"] = openrouter_api_key

In [ ]:
# Crear el modelo de generación
llm = ChatOpenRouter(
    model="google/gemini-2.5-flash-lite",
    temperature=0.3,  # Un poco más de creatividad para resúmenes interesantes
)

In [ ]:
# 3. Crear la aplicación FastAPI
app = FastAPI(
    title="Resumidor de texto extenso",
    description="API para resumir textos largos con control de longitud y estructura por secciones.",
    version="1.0.0",
)

print("✅ Entorno listo. Ahora definiremos los modelos y la lógica de resumen.")

En este proyecto, la entrada se envía como `text/plain` (no JSON), y el parámetro `summary_size` va en la URL. La salida sí es JSON.

In [ ]:
# 4. Modelos Pydantic
class SummaryContent(BaseModel):
    summary: str = Field(description="Resumen ejecutivo del texto.")
    key_points: List[str] = Field(
        description="Lista de entre 3 y 5 ideas clave."
    )
    conclusion: str = Field(description="Conclusión principal del texto.")


class SummarizeResponse(BaseModel):
    summary: str
    key_points: List[str]
    conclusion: str
    original_length: int
structured_llm = llm.with_structured_output(SummaryContent)

In [ ]:
# 5. Función para dividir texto largo
def split_text_for_summary(text: str, max_words: int = 5000):
    """
    Divide el texto si supera el límite de palabras.
    Retorna una lista de fragmentos o el texto original si es corto.
    """
    # Contar palabras (aproximado)
    word_count = len(text.split())

    if word_count <= max_words:
        # Texto corto: no dividir
        return [text]

    # Texto largo: dividir en fragmentos
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,       # Caracteres por fragmento
        chunk_overlap=200,     # Solapamiento entre fragmentos
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]  # Prioridad: párrafos, frases, palabras
    )
    chunks = splitter.split_text(text)
    return chunks

In [ ]:
# 6. Función para resumir un fragmento
def summarize_chunk(chunk: str) -> str:
    """Resume un fragmento de texto en 2-3 líneas."""
    prompt = f"""
Resume el siguiente texto en un máximo de 3 líneas. Sé conciso y captura lo esencial:

Texto: {chunk}

Resumen:
"""
    response = llm.invoke(prompt)
    return response.content.strip()

In [ ]:
# 7. Función para consolidar resúmenes
def consolidate_summaries(
    summaries: List[str],
    summary_size: str
) -> SummaryContent:
    """
    Consolida resúmenes parciales de forma jerárquica
    para evitar truncar información del documento.
    """

    length_instructions = {
        "short": "Muy breve, máximo 2 párrafos en total.",
        "medium": "Extensión media, entre 3 y 4 párrafos en total.",
        "long": "Detallado, entre 5 y 6 párrafos en total."
    }

    # Reducir progresivamente mientras haya demasiados resúmenes.
    current_summaries = summaries

    while len(current_summaries) > 8:
        reduced_summaries = []

        # Procesar grupos de hasta 8 resúmenes parciales
        for i in range(0, len(current_summaries), 8):
            group = current_summaries[i:i + 8]
            combined_group = "\n\n".join(group)

            prompt = f"""
Resume y consolida los siguientes resúmenes parciales.

Conserva toda la información importante y elimina repeticiones.

Resúmenes parciales:
---
{combined_group}
---

Devuelve un único resumen breve en español.
"""

            response = llm.invoke(prompt)
            reduced_summaries.append(response.content.strip())

        current_summaries = reduced_summaries

    combined = "\n\n".join(current_summaries)

    prompt = f"""
Eres un experto en análisis y síntesis de textos.

Genera el resumen final a partir de los siguientes resúmenes parciales.

El resultado debe contener:

- un resumen ejecutivo;
- entre 3 y 5 ideas clave;
- una conclusión.

Longitud deseada:
{length_instructions[summary_size]}

Resúmenes parciales:
---
{combined}
---

Responde en español.
"""

    return structured_llm.invoke(prompt)

Ahora unimos todas las piezas en el endpoint principal.

In [ ]:
# 8. Endpoint POST /summarize
@app.post("/summarize", response_model=SummarizeResponse)
def summarize(
    text: str = Body(..., media_type="text/plain"),
    summary_size: str = Query(
        "medium",
        pattern="^(short|medium|long)$"
    )
):
    """
    Recibe un texto en texto plano y devuelve un resumen estructurado.

    - text: texto a resumir, enviado como text/plain.
    - summary_size: short, medium o long.
    """

    try:
        # 1. Validar que el texto no esté vacío
        if not text or not text.strip():
            raise HTTPException(
                status_code=400,
                detail="El texto no puede estar vacío."
            )

        # 2. Dividir el texto si es necesario
        chunks = split_text_for_summary(text)

        length_instructions = {
            "short": "Muy breve, máximo 2 párrafos en total.",
            "medium": "Extensión media, entre 3 y 4 párrafos en total.",
            "long": "Detallado, entre 5 y 6 párrafos en total."
        }

        # 3. Texto corto: resumir directamente
        if len(chunks) == 1:
            prompt = f"""
Eres un experto en análisis y síntesis de textos.

Resume el siguiente texto.

El resultado debe contener:

- un resumen ejecutivo;
- entre 3 y 5 ideas clave;
- una conclusión.

Longitud deseada:
{length_instructions[summary_size]}

Texto:
---
{text}
---

Responde en español.
"""

            result = structured_llm.invoke(prompt)

        # 4. Texto largo: resumir fragmentos y consolidar
        else:
            print(
                f"📄 Texto dividido en {len(chunks)} fragmentos para resumir"
            )

            partial_summaries = []

            for i, chunk in enumerate(chunks):
                print(
                    f"   Resumiendo fragmento {i + 1}/{len(chunks)}..."
                )
                partial = summarize_chunk(chunk)
                partial_summaries.append(partial)

            print("🔄 Consolidando resúmenes parciales...")

            result = consolidate_summaries(
                partial_summaries,
                summary_size
            )

        # 5. Devolver respuesta
        return SummarizeResponse(
            summary=result.summary,
            key_points=result.key_points,
            conclusion=result.conclusion,
            original_length=len(text)
        )

    except HTTPException:
        raise

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Error al procesar el resumen: {str(e)}"
        )

In [ ]:
# 9. Levantar servidor
import threading
import time
import uvicorn

PORT = 8000

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()
time.sleep(2)
print(f"✅ Servidor iniciado en el puerto {PORT}")

In [ ]:
# 10. Abrir túnel con ngrok
from pyngrok import ngrok

ngrok_authtoken = userdata.get("NGROK_AUTHTOKEN")
if not ngrok_authtoken:
    raise ValueError("Agrega NGROK_AUTHTOKEN en los Secrets de Colab.")

ngrok.set_auth_token(ngrok_authtoken)
ngrok.kill()
tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

print("🌍 URL pública temporal:")
print(public_url)
print("\n📚 Documentación interactiva (Swagger UI):")
print(public_url + "/docs")